In [ ]:
import pandas as pd

df = pd.read_csv("./ia_tweets.csv")

df.head()

In [ ]:
df.dtypes

In [ ]:
from langdetect import detect, DetectorFactory

DetectorFactory.seed = 0

df['idioma'] = df['text'].apply(detect)

df = df[df["idioma"] == "es"]

df = df.drop(columns=["ID","user_has_extended_profile", "user_is_translator", "user_protected", "idioma"])

In [ ]:
def filtrar_por_polaridad(field):
    return df[df["polarity"] == field]

cant_negativos = len(filtrar_por_polaridad("N"))
cant_neutros = len(filtrar_por_polaridad("NEU"))
cant_positivos =  len(filtrar_por_polaridad("P"))

print('Cantidad de tweets: ', len(df))
print('Tweets Negativos: ', cant_negativos, ', Neutros' , cant_neutros,', Positivos', cant_positivos)

# Cantidad de datos desbalanceada, se tiene mucho mas datos neutros que negativos o positivos.

In [ ]:
cantidad_na = df.isna().sum()
print(cantidad_na)

#(cantidad_negativos_columna("favorite_count"))

# No hay nulls o valores

In [ ]:
cantidad_negativos = (df.select_dtypes(include='number') < 0).sum()
print(cantidad_negativos)

In [ ]:
import matplotlib.pyplot as plt

# Datos
clases = ['Positivo', 'Negativo', 'Neutral']
cantidades = [cant_positivos, cant_negativos, cant_neutros]

# Crear gráfico
plt.pie(cantidades, labels=clases, autopct='%1.1f%%')
plt.title('Distribución de sentimientos')
plt.show()

In [ ]:
print(df["favorite_count"].mean())
print(df["favorite_count"].max())
print(df["favorite_count"].min())

plt.scatter(range(0,len(df)), df['favorite_count'])

plt.xlabel('Ids')
plt.ylabel('Cantidad de me gusta')
plt.title('ID')

plt.show()

In [ ]:
print(df["retweet_count"].mean())
print(df["retweet_count"].max())
print(df["retweet_count"].min())

plt.scatter(range(0,len(df)), df['retweet_count'])

plt.xlabel('Id')
plt.ylabel('Cantidad de retweets')
plt.title('Retweets')

plt.show()

In [ ]:
df['tweet_interactions'] = df['favorite_count'] + df['retweet_count']

tweets_con_interaccion = (df['tweet_interactions'] > 0).sum()
tweets_sin_interaccion = (df['tweet_interactions'] == 0).sum()

positivos_con_interaccion = (df[df['polarity'] == 'P']['tweet_interactions'] > 1).sum()
neutros_con_interaccion = (df[df['polarity'] == 'NEU']['tweet_interactions'] > 1).sum()
negativos_con_interaccion = (df[df['polarity'] == 'N']['tweet_interactions'] > 1).sum()
interaccion_positivos_media = df[df['polarity'] == 'P']['tweet_interactions'].mean()
interaccion_negativos_media = df[df['polarity'] == 'N']['tweet_interactions'].mean()

print("Cantidad de tweets positivo con interaccion: ", positivos_con_interaccion)
print("Cantidad de tweets negativos con interaccion: ", negativos_con_interaccion)
print("Media de interacciones en positivos: ", interaccion_positivos_media)
print("Media de interacciones en negativos: ", interaccion_negativos_media)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))

ax[0].pie([positivos_con_interaccion, neutros_con_interaccion, negativos_con_interaccion], labels=["positivos", "neutros", "negativos"], autopct='%1.1f%%')
ax[0].set_title('Porcentaje de polaridades con interaccion')

ax[1].pie([tweets_con_interaccion, tweets_sin_interaccion],
  labels=["Interactuados", "No interactudos"], autopct='%1.1f%%')
ax[1].set_title('Porcentaje de tweets con interaccion')

plt.tight_layout()
plt.show()

In [ ]:
 #text
#polarity
#favorite_count
#retweet_count
#user_followers_count
#user_friends_count
#user_favourites_count         TEST (Likes que dio)
#user_statuses_count           TEST (Publicaciones y comentarios suyos)
#user_verified                 TEST
#user_default_profile          TEST (Pagina de perfil default)

import unicodedata

def contar_emojis(texto):
    return sum(1 for caracter in str(texto) if unicodedata.category(caracter) == 'So')

df["length"] = df['text'].apply(lambda texto: len(texto.split()))
df["hashtag_count"] = df['text'].apply(lambda texto: texto.count('#'))
df["mention_count"] = df['text'].apply(lambda texto: texto.count('@'))
df["url_count"] = df['text'].apply(lambda texto: texto.count('http'))
df["interrogation_count"] = df['text'].apply(lambda texto: texto.count('?') )
df["exclamation_count"] = df['text'].apply(lambda texto: texto.count('!'))
df['emoji_count'] = df['text'].apply(contar_emojis)

# :url: :mencion:

#Campos extra
#Longitud (palabras)
#Cantidad de hashtags
#Cantidad de usuarios mencionados
#Cantidad de emojis
#Cantidad de urls
#Cantidad de signos de exclamacion
#Cantidad de signos de interrogacion

In [ ]:
df["text"] = df["text"].str.replace(r"http\S+|www\.\S+", "URL", regex=True)
df["text"] = df["text"].str.replace(r"@\w+", "MENCION", regex=True)

df.head()

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["polarity"])
y = df["polarity"]

# 60% para train y 40% para test y validacion
X_train, X_test_val, y_train, y_test_val = train_test_split(
    X, y,
    test_size=0.40,
    random_state=42,
    stratify=y
)

# Segunda división: separar test y validación
X_test, X_val, y_test, y_val = train_test_split(
    X_test_val, y_test_val,
    test_size=0.5,
    random_state=42,
    stratify=y_test_val
)

print(len(X_train)/len(df))

In [ ]:
#from sklearn.preprocessing import StandardScaler

#number_columns = ["favorite_count","retweet_count","user_followers_count", "user_friends_count", "user_favourites_count", "user_statuses_count", "tweet_interactions", "hashtag_count", "mention_count"]

#scaler = StandardScaler()

#X_train[number_columns] = scaler.fit_transform(X_train[number_columns]) #No es necesario para RF pero lo dejo por consistencia

#y_train = pd.get_dummies(y_train, columns=["polarity"], dtype=int, drop_first=True)

#X_train.head()
#y_train.head()

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer, SnowballStemmer
import string

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

PUNCT = set(string.punctuation) | set("¿¡«»“”‘’…")
# Palabras que NO se eliminan aunque estén en la lista de stopwords
KEEP = {"no", "ni", "sin", "nunca", "tampoco", "nada", "contra", "muy", "pero", "más", "menos"}
STOP_WORDS = set(stopwords.words('spanish')) - KEEP

class TextCleaner(BaseEstimator, TransformerMixin):
    def __init__(self, method=None, ngram_range=(1, 1), remove_stopwords=True):
        self.method = method                      # None, "stem" o "lemma"
        self.ngram_range = ngram_range            # (1, 1) = solo palabras sueltas
        self.remove_stopwords = remove_stopwords  # True = saca stopwords (salvo KEEP)

    def fit(self, X, y=None):
        if self.method not in (None, "stem", "lemma"):
            raise ValueError("method debe ser None, 'stem' o 'lemma'")
        return self

    def _is_valid(self, token):
        if not token.strip() or all(c in PUNCT for c in token):
            return False
        if self.remove_stopwords and token in STOP_WORDS:
            return False
        return True

    def _ngrams(self, tokens):
        n_min, n_max = self.ngram_range
        out = []
        for n in range(n_min, n_max + 1):
            out += ["_".join(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]
        return out

    def transform(self, X):
        X = list(X)

        if self.method == "lemma":
            import spacy
            nlp = spacy.load("es_core_news_sm", disable=["parser", "ner"])
            token_lists = []
            for doc in nlp.pipe(X):
                # se filtra por el texto original y después se toma el lema
                token_lists.append([t.lemma_.lower() for t in doc if self._is_valid(t.text.lower())])
        else:
            token_lists = [
                [t for t in nltk.word_tokenize(text.lower(), language="spanish") if self._is_valid(t)]
                for text in X
            ]
            if self.method == "stem":
                stemmer = SnowballStemmer("spanish")
                token_lists = [[stemmer.stem(t) for t in toks] for toks in token_lists]

        return [" ".join(self._ngrams(toks)) for toks in token_lists]

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

cleaner = TextCleaner("lemma")

vectores = cleaner.transform(df["text"])
vectorizer_BoW = CountVectorizer()
vectorizer_Tfidf = TfidfVectorizer()


matriz_BoW = vectorizer_BoW.fit_transform(vectores)
matriz_Tfidf = vectorizer_Tfidf.fit_transform(vectores)

print(vectorizer_BoW.get_feature_names_out())
print(matriz_BoW.toarray()[0])
#print(vectores_Tfidf.shape)

In [ ]:
from sentence_transformers import SentenceTransformer # <- Estandar para Embeddings

class EmbeddingTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        self.model_name = model_name
        self.model = None

    def fit(self, X, y=None):
        # Cargamos el modelo aquí para evitar errores de serialización
        if self.model is None:
            self.model = SentenceTransformer(self.model_name)
        return self

    def transform(self, X):
        # Convertimos la lista de textos en una matriz de vectores (Embeddings)
        # Esto implementa implícitamente el "Mean Pooling"
        return self.model.encode(list(X), show_progress_bar=False)

embedding = EmbeddingTransformer()

In [ ]:
embedding.fit(df["text"])
text = embedding.transform(df["text"])

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import clone

num_cols = [
    "favorite_count", "retweet_count", "user_followers_count",
    "user_friends_count", "user_favourites_count", "user_statuses_count",
    "tweet_interactions", "hashtag_count", "mention_count", "url_count",
    "exclamation_count", "interrogation_count", "length", "emoji_count"
]

preprocesador = ColumnTransformer([
    ("text", Pipeline([
        ("cleaner", TextCleaner()),
        ("vectorizer", CountVectorizer())
    ]), "text"),
    ("num", StandardScaler(), num_cols) #En teoria no hacen falta en RF pero lo pruebo
])

 ### Principales hiperparámetros de RandomForestClassifier

| Hiperparámetro | Valor por default | ¿Por qué afecta al rendimiento? |
|---|---|---|
| `max_depth` | None | La profundidad máxima del árbol. Si es None, los nodos se expanden hasta que todas las hojas sean puras o hasta que todas las hojas contengan menos de min_samples_split muestras. |
| `min_samples_leaf` | 1 | El número mínimo de muestras requerido en un nodo hoja. Un punto de división a cualquier profundidad solo se considerará si deja al menos `min_samples_leaf` muestras de entrenamiento tanto en la rama izquierda como en la derecha. Esto puede tener el efecto de suavizar el modelo, especialmente en problemas de regresión. |
| `min_samples_split` | 2 | El número mínimo de muestras requerido para dividir un nodo interno. |
| `criterion` | `gini` | La función para medir la calidad de una división. Los criterios admitidos son «gini» (para la impureza de Gini) y «log_loss» o «entropy» (ambos para la ganancia de información de Shannon); véase la formulación matemática. |
| `max_features` | `sqrt` | La cantidad de características a tener en cuenta al buscar la mejor división. |
| `max_leaf_nodes` | None | Cultiva árboles con max_leaf_nodes en orden de mejor a mejor. Los mejores nodos se definen como la reducción relativa de impurezas. Si es None, el número de nodos hoja es ilimitado. |

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

# El Pipeline combina el vectorizador y el clasificador
pipeline_Bow_y_TFidf = Pipeline([
    ("pre", clone(preprocesador)),
    ('classifier', RandomForestClassifier(random_state=42))
])

# Parámetros a tunear
param_grid = [
    # Opción 3: Random Forest
    {
        'pre__text__cleaner__method': [None, 'stem', 'lemma'],
        'pre__text__vectorizer': [CountVectorizer(), TfidfVectorizer()],
        'pre__text__cleaner__ngram_range': [(1,1), (1,2)],
        'classifier': [RandomForestClassifier(random_state=42)],
        'classifier__n_estimators': [100, 200, 300],
        'classifier__criterion': ['gini', 'entropy', 'log_loss'],
        #'classifier__max_depth': [None, 10, 20],
        'classifier__min_samples_split': [2, 5],
        'classifier__min_samples_leaf': [1, 2],
        'classifier__max_features': ['sqrt', 'log2'],
        #'classifier__max_leaf_nodes': [None, 50, 100],
        #'classifier__min_impurity_decrease': [0.0, 0.001, 0.01],
        #'classifier__ccp_alpha': [0.0, 0.001, 0.01],
        #'classifier__min_weight_fraction_leaf': [0.0, 0.01, 0.05],
        'classifier__class_weight': [None, 'balanced'],
    },

]

In [ ]:
# Ejecución del Grid Search optimizando ROC
grid_search_rf = GridSearchCV(pipeline_Bow_y_TFidf, param_grid, cv=2, scoring='roc_auc_ovr', n_jobs=3, verbose=1)
grid_search_rf.fit(X_train, y_train)
print("Mejores parámetros:", grid_search_rf.best_params_)
print("Mejor ROC:", grid_search_rf.best_score_)

In [ ]:
from sklearn.metrics import (
    f1_score,
    roc_auc_score,
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay
)

import matplotlib.pyplot as plt

mejor_modelo_rf = grid_search_rf.best_estimator_

y_pred_rf = mejor_modelo_rf.predict(X_val)

print("Forma de y_train:", y_train.shape)
print("Tipo de y_train:", type(y_train))
print("Primeras etiquetas:")
print(y_train[:5])

print("Clases originales:")
print(df["polarity"].value_counts())


f1_rf = f1_score(
    y_val,
    y_pred_rf,
    average="macro"
)

accuracy_rf = accuracy_score(y_val, y_pred_rf)

print(f"F1 macro: {f1_rf:.4f}")
print(f"Accuracy: {accuracy_rf:.4f}")

print(
    classification_report(
        y_val,
        y_pred_rf,
        labels=["N", "NEU", "P"],
        digits=4,
        zero_division=0
    )
)

ConfusionMatrixDisplay.from_predictions(
    y_val,
    y_pred_rf,
    labels=["N", "NEU", "P"],
    display_labels=["Negativo", "Neutral", "Positivo"],
    cmap="Blues"
)

plt.show()

In [ ]:
preprocesador_embedding = ColumnTransformer([
    ("text", EmbeddingTransformer(), "text"),
    ("num", StandardScaler(), num_cols),
])

pipeline_embedding = Pipeline([
    ("pre", preprocesador_embedding),
    ("classifier", RandomForestClassifier(
        random_state=42,
        n_jobs=1
    ))
])

param_grid_embedding = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__criterion': ['gini', 'entropy', 'log_loss'],
    #'classifier__max_depth': [None, 10, 20],
    'classifier__min_samples_split': [2, 5],
    'classifier__min_samples_leaf': [1, 2],
    'classifier__max_features': ['sqrt', 'log2'],
    #'classifier__max_leaf_nodes': [None, 50, 100],
    #'classifier__min_impurity_decrease': [0.0, 0.001, 0.01],
    #'classifier__ccp_alpha': [0.0, 0.001, 0.01],
    #'classifier__min_weight_fraction_leaf': [0.0, 0.01, 0.05],
    'classifier__class_weight': [None, 'balanced']
}

grid_search_rf_embedding = GridSearchCV(pipeline_embedding, param_grid_embedding, cv=3, scoring="roc_auc_ovr", n_jobs=3, verbose=1)

grid_search_rf_embedding.fit(X_train, y_train)

print("-" * 30)
print(
    "Mejor ROC entrenamiento:",
    f"{grid_search_rf_embedding.best_score_:.4f}"
)
print("Mejores parámetros:")
print(grid_search_rf_embedding.best_params_)
print("-" * 30)

In [ ]:
mejor_rf_embedding = grid_search_rf_embedding.best_estimator_

y_pred_rf_embedding = mejor_rf_embedding.predict(X_val)

#print("ROC:", roc_auc_score(y_val, y_pred_rf_embedding, multi_class="ovr", average="macro"))

print(classification_report(y_val, y_pred_rf_embedding, labels=["N", "NEU", "P"], digits=4, zero_division=0))